# Chapter 7: Scientific Research Automation and AI Scientists

## Complete Code Implementations

This notebook contains all code listings from Chapter 7, covering AI agents for scientific research automation including:

- **7.1** Hypothesis Generation Agent
- **7.2** Materials Property Prediction Pipeline
- **7.3** Autonomous Experiment Execution Workflow
- **7.4** Scientific Knowledge Graph Construction Agent
- **7.5** Cross-Disciplinary Research Coordinator

---

## Setup and Dependencies

Install required packages before running the examples.

In [ ]:
# Install dependencies
# !pip install langgraph langchain langchain-openai pydantic numpy

In [ ]:
# Common imports used across listings
from typing import TypedDict, Optional, List, Dict, Any, Literal, Annotated
from datetime import datetime
from enum import Enum
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
import json
import uuid
import random
from collections import deque

---

## Listing 7-1: Hypothesis Generation Agent

A LangGraph StateGraph agent that mines literature, identifies knowledge gaps, generates candidate hypotheses, and ranks them by novelty and feasibility. All external API calls are simulated so the notebook runs without credentials.

In [ ]:
"""
Listing 7-1: Hypothesis Generation Agent

Demonstrates a phased hypothesis generation workflow using LangGraph.
All tool calls return simulated data so the notebook is self-contained.
"""

from typing import TypedDict, Annotated, List, Dict, Any
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json, random, uuid


# ------------------------------------------------------------------ #
# State                                                               #
# ------------------------------------------------------------------ #

class HypothesisState(TypedDict):
    """State for hypothesis generation workflow."""
    messages: Annotated[list[BaseMessage], add_messages]
    research_domain: str
    seed_topic: str
    literature_findings: list[dict]
    knowledge_gaps: list[str]
    candidate_hypotheses: list[dict]
    ranked_hypotheses: list[dict]
    current_phase: str
    iteration_count: int
    max_iterations: int


# ------------------------------------------------------------------ #
# Simulated Tools                                                     #
# ------------------------------------------------------------------ #

def search_literature(query: str, domain: str, max_results: int = 20) -> list[dict]:
    """Simulated literature search returning mock papers."""
    mock_papers = [
        {
            "title": f"Advances in {domain}: {query} – A Comprehensive Review",
            "authors": ["Chen, L.", "Wang, X.", "Zhang, Y."],
            "year": 2024,
            "venue": "Nature Materials",
            "citation_count": 142,
            "abstract": (
                f"We review recent progress in {query} within {domain}, "
                "identifying key trends and open questions."
            ),
            "key_findings": [
                f"Novel {query} mechanisms discovered in 2023-2024",
                "Computational methods achieve 90%+ accuracy for property prediction",
                "Gap: limited experimental validation of ML predictions",
            ],
        },
        {
            "title": f"Machine Learning for {query} in {domain}",
            "authors": ["Smith, J.", "Lee, K."],
            "year": 2023,
            "venue": "Science",
            "citation_count": 89,
            "abstract": (
                f"We demonstrate ML approaches to {query}, showing "
                "significant improvements over traditional methods."
            ),
            "key_findings": [
                "Transfer learning reduces data requirements by 60%",
                "Graph neural networks outperform descriptor-based models",
                "Gap: interpretability of ML models remains limited",
            ],
        },
        {
            "title": f"Experimental Validation of {query} Predictions",
            "authors": ["Park, S.", "Johnson, R.", "Kim, H."],
            "year": 2024,
            "venue": "Advanced Materials",
            "citation_count": 56,
            "abstract": (
                f"We experimentally validate computational predictions for {query}, "
                "revealing both successes and systematic discrepancies."
            ),
            "key_findings": [
                "67% of computational predictions confirmed experimentally",
                "Systematic bias in formation energy predictions for oxides",
                "Gap: need for uncertainty quantification in predictions",
            ],
        },
    ]
    return mock_papers[:max_results]


def analyze_gaps(literature: list[dict], domain: str) -> list[str]:
    """Simulated gap analysis based on literature findings."""
    gaps = [
        f"Limited experimental validation of ML-predicted properties in {domain}",
        f"Lack of uncertainty quantification in {domain} prediction models",
        f"Poor interpretability of deep learning models for {domain}",
        f"Insufficient exploration of multi-component systems in {domain}",
        f"No standardized benchmarks for {domain} prediction accuracy",
    ]
    return gaps


def evaluate_novelty(hypothesis: str, domain: str) -> dict:
    """Simulated novelty evaluation for a hypothesis."""
    score = round(random.uniform(0.5, 0.95), 2)
    return {
        "novelty_score": score,
        "feasibility_score": round(random.uniform(0.4, 0.9), 2),
        "impact_score": round(random.uniform(0.6, 0.95), 2),
        "similar_work_count": random.randint(0, 5),
        "assessment": "novel" if score > 0.7 else "incremental",
    }


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def search_literature_node(state: HypothesisState) -> dict:
    """Phase 1: Mine the literature for the seed topic."""
    domain = state["research_domain"]
    topic = state["seed_topic"]
    papers = search_literature(topic, domain)
    return {
        "literature_findings": papers,
        "current_phase": "literature_mining",
        "messages": [AIMessage(content=f"Found {len(papers)} relevant papers for '{topic}' in {domain}.")],
    }


def identify_gaps_node(state: HypothesisState) -> dict:
    """Phase 2: Identify knowledge gaps from the literature."""
    gaps = analyze_gaps(state["literature_findings"], state["research_domain"])
    return {
        "knowledge_gaps": gaps,
        "current_phase": "gap_analysis",
        "messages": [AIMessage(content=f"Identified {len(gaps)} knowledge gaps.")],
    }


def generate_hypotheses_node(state: HypothesisState) -> dict:
    """Phase 3: Generate candidate hypotheses addressing the gaps."""
    domain = state["research_domain"]
    gaps = state["knowledge_gaps"]
    hypotheses = []
    for i, gap in enumerate(gaps[:3]):
        hypotheses.append({
            "id": f"H-{i+1}",
            "statement": (
                f"Hypothesis {i+1}: Addressing '{gap}' through a combined "
                f"computational-experimental approach in {domain} will yield "
                "statistically significant improvements in prediction accuracy."
            ),
            "addresses_gap": gap,
            "proposed_method": f"Multi-modal learning with experimental feedback loop",
            "estimated_timeline_months": random.randint(6, 18),
        })
    return {
        "candidate_hypotheses": hypotheses,
        "current_phase": "hypothesis_generation",
        "messages": [AIMessage(content=f"Generated {len(hypotheses)} candidate hypotheses.")],
    }


def evaluate_and_rank_node(state: HypothesisState) -> dict:
    """Phase 4: Evaluate novelty/feasibility and rank hypotheses."""
    domain = state["research_domain"]
    ranked = []
    for hyp in state["candidate_hypotheses"]:
        scores = evaluate_novelty(hyp["statement"], domain)
        ranked.append({
            **hyp,
            **scores,
            "composite_score": round(
                0.4 * scores["novelty_score"]
                + 0.3 * scores["feasibility_score"]
                + 0.3 * scores["impact_score"], 3
            ),
        })
    ranked.sort(key=lambda h: h["composite_score"], reverse=True)
    return {
        "ranked_hypotheses": ranked,
        "current_phase": "ranking_complete",
        "messages": [AIMessage(content="Hypotheses ranked. Workflow complete.")],
    }


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_hypothesis_generation_graph():
    """Build the phased hypothesis generation workflow."""
    workflow = StateGraph(HypothesisState)
    workflow.add_node("search_literature", search_literature_node)
    workflow.add_node("identify_gaps", identify_gaps_node)
    workflow.add_node("generate_hypotheses", generate_hypotheses_node)
    workflow.add_node("evaluate_and_rank", evaluate_and_rank_node)

    workflow.set_entry_point("search_literature")
    workflow.add_edge("search_literature", "identify_gaps")
    workflow.add_edge("identify_gaps", "generate_hypotheses")
    workflow.add_edge("generate_hypotheses", "evaluate_and_rank")
    workflow.add_edge("evaluate_and_rank", END)

    return workflow.compile()


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

graph = build_hypothesis_generation_graph()

initial_state = {
    "messages": [HumanMessage(content="Generate hypotheses for perovskite solar cell stability")],
    "research_domain": "materials science",
    "seed_topic": "perovskite solar cell degradation mechanisms",
    "literature_findings": [],
    "knowledge_gaps": [],
    "candidate_hypotheses": [],
    "ranked_hypotheses": [],
    "current_phase": "initialized",
    "iteration_count": 0,
    "max_iterations": 3,
}

result = graph.invoke(initial_state)

print("=" * 70)
print("HYPOTHESIS GENERATION RESULTS")
print("=" * 70)
print(f"\nDomain: {result['research_domain']}")
print(f"Topic:  {result['seed_topic']}")
print(f"\nLiterature papers found: {len(result['literature_findings'])}")
print(f"Knowledge gaps identified: {len(result['knowledge_gaps'])}")
print(f"\nRanked Hypotheses:")
for h in result["ranked_hypotheses"]:
    print(f"\n  [{h['id']}] Composite Score: {h['composite_score']}")
    print(f"      Novelty: {h['novelty_score']}  Feasibility: {h['feasibility_score']}  Impact: {h['impact_score']}")
    print(f"      Statement: {h['statement'][:100]}...")


---

## Listing 7-2: Materials Property Prediction Pipeline

Ensemble prediction of materials properties (bandgap, formation energy, synthesizability) with screening and analysis. All predictions are simulated.

In [ ]:
"""
Listing 7-2: Materials Property Prediction Pipeline

LangGraph workflow that screens candidate materials using multiple
simulated ML property predictors and produces a ranked recommendation.
"""

from typing import TypedDict, Annotated, List, Dict, Any, Optional
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json, random, uuid


# ------------------------------------------------------------------ #
# State                                                               #
# ------------------------------------------------------------------ #

class MaterialsState(TypedDict):
    """State for materials property prediction workflow."""
    messages: Annotated[list[BaseMessage], add_messages]
    candidate_materials: list[dict]
    predictions: list[dict]
    screening_results: list[dict]
    analysis: dict
    phase: str


# ------------------------------------------------------------------ #
# Simulated Property Prediction Tools                                 #
# ------------------------------------------------------------------ #

def predict_properties(formula: str) -> dict:
    """Simulated ML property prediction for a material formula."""
    base_bg = hash(formula) % 40 / 10.0  # deterministic but varied
    return {
        "formula": formula,
        "bandgap_eV": round(base_bg + random.uniform(-0.3, 0.3), 2),
        "bandgap_uncertainty_eV": round(random.uniform(0.05, 0.2), 3),
        "formation_energy_eV_per_atom": round(random.uniform(-3.5, -0.2), 3),
        "formation_energy_uncertainty": round(random.uniform(0.01, 0.1), 3),
        "is_thermodynamically_stable": random.choice([True, True, False]),
        "distance_to_hull_eV": round(random.uniform(0.0, 0.15), 4),
        "model": "MEGNet-v2 + CGCNN ensemble",
        "confidence": round(random.uniform(0.7, 0.98), 2),
    }


def screen_candidates(predictions: list[dict],
                       min_bandgap: float = 1.0,
                       max_bandgap: float = 3.0,
                       max_hull_distance: float = 0.1) -> list[dict]:
    """Screen candidates by property thresholds."""
    results = []
    for p in predictions:
        passed = (
            min_bandgap <= p["bandgap_eV"] <= max_bandgap
            and p["distance_to_hull_eV"] <= max_hull_distance
            and p["is_thermodynamically_stable"]
        )
        results.append({
            **p,
            "passed_screening": passed,
            "fail_reasons": [] if passed else _get_fail_reasons(p, min_bandgap, max_bandgap, max_hull_distance),
        })
    return results


def _get_fail_reasons(p, min_bg, max_bg, max_hull):
    reasons = []
    if p["bandgap_eV"] < min_bg or p["bandgap_eV"] > max_bg:
        reasons.append(f"Bandgap {p['bandgap_eV']} eV outside range [{min_bg}, {max_bg}]")
    if p["distance_to_hull_eV"] > max_hull:
        reasons.append(f"Hull distance {p['distance_to_hull_eV']} eV > {max_hull}")
    if not p["is_thermodynamically_stable"]:
        reasons.append("Thermodynamically unstable")
    return reasons


def analyze_stability(candidate: dict) -> dict:
    """Simulated stability assessment for a candidate material."""
    return {
        "formula": candidate["formula"],
        "thermal_stability": random.choice(["high", "moderate", "low"]),
        "mechanical_stability": random.choice(["stable", "marginally stable"]),
        "chemical_stability": random.choice(["inert", "reactive in moisture", "air-stable"]),
        "overall_assessment": random.choice(["promising", "needs further study", "not recommended"]),
        "recommended_next_steps": [
            "DFT phonon calculation for dynamic stability",
            "Molecular dynamics at 300K, 500K, 800K",
            "Experimental synthesis attempt via solid-state reaction",
        ],
    }


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def predict_node(state: MaterialsState) -> dict:
    """Run property predictions on all candidate materials."""
    predictions = [predict_properties(m["formula"]) for m in state["candidate_materials"]]
    return {
        "predictions": predictions,
        "phase": "prediction_complete",
        "messages": [AIMessage(content=f"Predicted properties for {len(predictions)} materials.")],
    }


def screen_node(state: MaterialsState) -> dict:
    """Screen candidates against target property windows."""
    screened = screen_candidates(state["predictions"])
    passed = [s for s in screened if s["passed_screening"]]
    return {
        "screening_results": screened,
        "phase": "screening_complete",
        "messages": [AIMessage(content=f"Screening complete: {len(passed)}/{len(screened)} candidates passed.")],
    }


def analyze_node(state: MaterialsState) -> dict:
    """Detailed stability analysis of passed candidates."""
    passed = [s for s in state["screening_results"] if s["passed_screening"]]
    analyses = [analyze_stability(c) for c in passed]

    promising = [a for a in analyses if a["overall_assessment"] == "promising"]
    summary = {
        "total_candidates": len(state["candidate_materials"]),
        "passed_screening": len(passed),
        "promising_candidates": len(promising),
        "detailed_analyses": analyses,
        "top_recommendation": promising[0] if promising else (analyses[0] if analyses else None),
    }
    return {
        "analysis": summary,
        "phase": "analysis_complete",
        "messages": [AIMessage(content=f"Analysis complete. {len(promising)} promising candidates identified.")],
    }


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_materials_prediction_graph():
    """Build ensemble screening -> analysis workflow."""
    workflow = StateGraph(MaterialsState)
    workflow.add_node("predict", predict_node)
    workflow.add_node("screen", screen_node)
    workflow.add_node("analyze", analyze_node)

    workflow.set_entry_point("predict")
    workflow.add_edge("predict", "screen")
    workflow.add_edge("screen", "analyze")
    workflow.add_edge("analyze", END)

    return workflow.compile()


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

graph = build_materials_prediction_graph()

initial_state = {
    "messages": [HumanMessage(content="Screen candidate perovskite materials for solar cell applications")],
    "candidate_materials": [
        {"formula": "CsPbI3", "type": "halide perovskite"},
        {"formula": "MAPbBr3", "type": "hybrid perovskite"},
        {"formula": "FAPbI3", "type": "hybrid perovskite"},
        {"formula": "CsSnI3", "type": "lead-free perovskite"},
        {"formula": "Cs2AgBiBr6", "type": "double perovskite"},
        {"formula": "BaZrS3", "type": "chalcogenide perovskite"},
    ],
    "predictions": [],
    "screening_results": [],
    "analysis": {},
    "phase": "initialized",
}

result = graph.invoke(initial_state)

print("=" * 70)
print("MATERIALS PROPERTY PREDICTION RESULTS")
print("=" * 70)
analysis = result["analysis"]
print(f"\nTotal candidates screened: {analysis['total_candidates']}")
print(f"Passed screening: {analysis['passed_screening']}")
print(f"Promising candidates: {analysis['promising_candidates']}")
if analysis.get("top_recommendation"):
    rec = analysis["top_recommendation"]
    print(f"\nTop recommendation: {rec['formula']}")
    print(f"  Thermal stability: {rec['thermal_stability']}")
    print(f"  Mechanical stability: {rec['mechanical_stability']}")
    print(f"  Chemical stability: {rec['chemical_stability']}")
    print(f"  Assessment: {rec['overall_assessment']}")
else:
    print("\nNo candidates met all screening criteria.")


---

## Listing 7-3: Autonomous Experiment Execution Workflow

An iterative design-execute-analyze loop with safety verification. Simulates laboratory equipment interactions and supports multiple experimental iterations with a conditional continuation edge.

In [ ]:
"""
Listing 7-3: Autonomous Experiment Execution Workflow

LangGraph workflow implementing an iterative experiment loop with
safety verification, execution, analysis, and conditional continuation.
All laboratory interactions are simulated.
"""

from typing import TypedDict, Annotated, List, Dict, Any
from enum import Enum
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json, random, uuid
from datetime import datetime


# ------------------------------------------------------------------ #
# Enums & State                                                       #
# ------------------------------------------------------------------ #

class SafetyLevel(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"


class ExperimentState(TypedDict):
    """State for autonomous experiment execution."""
    messages: Annotated[list[BaseMessage], add_messages]
    experiment_id: str
    hypothesis: str
    experiment_plan: dict
    safety_status: str
    preparation_status: str
    execution_log: list[dict]
    results: dict
    iteration: int
    max_iterations: int
    phase: str
    requires_human_approval: bool


# ------------------------------------------------------------------ #
# Simulated Laboratory Tools                                          #
# ------------------------------------------------------------------ #

def check_safety(operation: str, reagents: list[str],
                 temperature_c: float, pressure_atm: float) -> dict:
    """Simulated safety verification with tiered approval."""
    risk_score = 0.0
    reasons = []
    if temperature_c > 500:
        risk_score += 0.3
        reasons.append(f"High temperature ({temperature_c} C)")
    if pressure_atm > 5:
        risk_score += 0.3
        reasons.append(f"Elevated pressure ({pressure_atm} atm)")
    if any("hazardous" in r.lower() for r in reagents):
        risk_score += 0.4
        reasons.append("Hazardous reagent detected")

    if risk_score > 0.6:
        level = SafetyLevel.HIGH
    elif risk_score > 0.3:
        level = SafetyLevel.MEDIUM
    else:
        level = SafetyLevel.LOW

    return {
        "approved": level != SafetyLevel.HIGH,
        "safety_level": level.value,
        "risk_score": round(risk_score, 2),
        "warnings": reasons,
        "required_ppe": ["lab coat", "safety glasses"]
            + (["fume hood"] if temperature_c > 200 else []),
        "emergency_protocol": "standard" if level == SafetyLevel.LOW else "enhanced",
    }


def prepare_experiment(plan: dict) -> dict:
    """Simulated experiment preparation status."""
    return {
        "preparation_id": str(uuid.uuid4())[:8],
        "status": "ready",
        "equipment_calibrated": True,
        "reagents_verified": True,
        "estimated_duration_hours": round(random.uniform(2, 8), 1),
        "setup_notes": [
            f"Furnace pre-heated to {plan.get('temperature_c', 800)} C",
            "Crucibles cleaned and dried",
            "Inert atmosphere (Ar) established",
        ],
    }


def execute_experiment(plan: dict, iteration: int) -> dict:
    """Simulated experiment execution returning measurement data."""
    base_yield = 0.5 + iteration * 0.1  # Improves with iterations
    return {
        "execution_id": str(uuid.uuid4())[:8],
        "timestamp": datetime.now().isoformat(),
        "measurements": {
            "yield_percent": round(min(base_yield + random.uniform(-0.1, 0.15), 0.99) * 100, 1),
            "phase_purity_percent": round(random.uniform(85, 99), 1),
            "crystallite_size_nm": round(random.uniform(20, 100), 1),
            "xrd_peaks_matched": random.randint(8, 15),
        },
        "observations": [
            "Color change observed at t=30 min (white -> pale yellow)",
            "Exothermic event detected at 650 C",
            "Final product: fine powder, homogeneous appearance",
        ],
        "status": "completed",
    }


def analyze_results(measurements: dict, iteration: int) -> dict:
    """Simulated statistical analysis of experimental results."""
    yield_val = measurements["yield_percent"]
    purity = measurements["phase_purity_percent"]
    quality_score = round((yield_val / 100 * 0.5 + purity / 100 * 0.5), 3)
    return {
        "quality_score": quality_score,
        "yield_assessment": "acceptable" if yield_val > 60 else "low",
        "purity_assessment": "high" if purity > 95 else "moderate",
        "statistical_confidence": round(random.uniform(0.85, 0.99), 3),
        "meets_threshold": quality_score > 0.8,
        "suggested_adjustments": (
            []
            if quality_score > 0.8
            else [
                "Increase sintering temperature by 50 C",
                "Extend hold time by 1 hour",
                "Reduce heating rate to 2 C/min",
            ]
        ),
        "recommendation": "accept" if quality_score > 0.8 else "iterate",
    }


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def safety_check_node(state: ExperimentState) -> dict:
    """Run safety verification on the experiment plan."""
    plan = state["experiment_plan"]
    safety = check_safety(
        operation=plan.get("operation", "synthesis"),
        reagents=plan.get("reagents", []),
        temperature_c=plan.get("temperature_c", 800),
        pressure_atm=plan.get("pressure_atm", 1),
    )
    return {
        "safety_status": "approved" if safety["approved"] else "rejected",
        "requires_human_approval": not safety["approved"],
        "phase": "safety_checked",
        "messages": [AIMessage(content=(
            f"Safety check: {safety['safety_level']} risk. "
            f"{'Approved' if safety['approved'] else 'REQUIRES HUMAN APPROVAL'}. "
            f"Warnings: {', '.join(safety['warnings']) or 'None'}"
        ))],
    }


def prepare_node(state: ExperimentState) -> dict:
    """Prepare laboratory equipment and reagents."""
    prep = prepare_experiment(state["experiment_plan"])
    return {
        "preparation_status": prep["status"],
        "phase": "prepared",
        "messages": [AIMessage(content=(
            f"Preparation complete. Estimated duration: {prep['estimated_duration_hours']}h. "
            f"Equipment calibrated: {prep['equipment_calibrated']}"
        ))],
    }


def execute_node(state: ExperimentState) -> dict:
    """Execute the experiment and collect measurements."""
    data = execute_experiment(state["experiment_plan"], state["iteration"])
    log = state.get("execution_log", [])
    log.append(data)
    return {
        "results": data["measurements"],
        "execution_log": log,
        "phase": "executed",
        "messages": [AIMessage(content=(
            f"Experiment executed (iteration {state['iteration'] + 1}). "
            f"Yield: {data['measurements']['yield_percent']}%, "
            f"Purity: {data['measurements']['phase_purity_percent']}%"
        ))],
    }


def analyze_node(state: ExperimentState) -> dict:
    """Analyze results and decide whether to continue."""
    analysis = analyze_results(state["results"], state["iteration"])
    return {
        "results": {**state["results"], "analysis": analysis},
        "iteration": state["iteration"] + 1,
        "phase": "analyzed",
        "messages": [AIMessage(content=(
            f"Analysis: quality={analysis['quality_score']}, "
            f"recommendation={analysis['recommendation']}"
        ))],
    }


def should_continue(state: ExperimentState) -> str:
    """Conditional edge: continue iterating or end."""
    analysis = state["results"].get("analysis", {})
    if analysis.get("meets_threshold", False):
        return "end"
    if state["iteration"] >= state["max_iterations"]:
        return "end"
    return "continue"


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_experiment_workflow():
    """Build iterative experiment execution workflow."""
    workflow = StateGraph(ExperimentState)
    workflow.add_node("safety_check", safety_check_node)
    workflow.add_node("prepare", prepare_node)
    workflow.add_node("execute", execute_node)
    workflow.add_node("analyze", analyze_node)

    workflow.set_entry_point("safety_check")
    workflow.add_edge("safety_check", "prepare")
    workflow.add_edge("prepare", "execute")
    workflow.add_edge("execute", "analyze")
    workflow.add_conditional_edges(
        "analyze",
        should_continue,
        {"continue": "execute", "end": END},
    )

    return workflow.compile()


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

graph = build_experiment_workflow()

initial_state = {
    "messages": [HumanMessage(content="Synthesize BaZrS3 thin film via sulfurization")],
    "experiment_id": str(uuid.uuid4())[:8],
    "hypothesis": "BaZrS3 can be synthesized with >90% phase purity via sulfurization at 900C",
    "experiment_plan": {
        "operation": "thin film sulfurization",
        "reagents": ["BaCO3", "ZrO2", "CS2"],
        "temperature_c": 900,
        "pressure_atm": 1,
        "duration_hours": 4,
        "atmosphere": "Ar + CS2",
    },
    "safety_status": "",
    "preparation_status": "",
    "execution_log": [],
    "results": {},
    "iteration": 0,
    "max_iterations": 3,
    "phase": "initialized",
    "requires_human_approval": False,
}

result = graph.invoke(initial_state)

print("=" * 70)
print("AUTONOMOUS EXPERIMENT RESULTS")
print("=" * 70)
print(f"\nExperiment ID: {result['experiment_id']}")
print(f"Hypothesis: {result['hypothesis']}")
print(f"Iterations completed: {result['iteration']}")
print(f"Safety status: {result['safety_status']}")
print(f"\nFinal measurements:")
for k, v in result["results"].items():
    if k != "analysis":
        print(f"  {k}: {v}")
if "analysis" in result["results"]:
    a = result["results"]["analysis"]
    print(f"\nQuality score: {a['quality_score']}")
    print(f"Recommendation: {a['recommendation']}")


---

## Listing 7-4: Scientific Knowledge Graph Construction Agent

Builds a scientific knowledge graph from text by extracting entities and relationships, validating them, and constructing an in-memory graph with BFS path finding.

In [ ]:
"""
Listing 7-4: Scientific Knowledge Graph Construction Agent

Extracts entities and relationships from scientific text, validates them,
and builds an in-memory knowledge graph with query and path-finding
capabilities. All extraction is simulated.
"""

from typing import TypedDict, Annotated, List, Dict, Any, Optional
from dataclasses import dataclass, field
from collections import deque
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json, uuid


# ------------------------------------------------------------------ #
# Knowledge Graph Data Structures                                     #
# ------------------------------------------------------------------ #

@dataclass
class KnowledgeEntity:
    """A node in the scientific knowledge graph."""
    entity_id: str
    entity_type: str   # material, property, method, instrument, paper
    name: str
    attributes: dict = field(default_factory=dict)
    confidence: float = 1.0
    provenance: list = field(default_factory=list)


@dataclass
class KnowledgeRelation:
    """An edge in the scientific knowledge graph."""
    source_id: str
    target_id: str
    relation_type: str  # has_property, synthesized_from, measured_by, etc.
    attributes: dict = field(default_factory=dict)
    confidence: float = 1.0
    evidence: list = field(default_factory=list)


class ScientificKnowledgeGraph:
    """In-memory knowledge graph with entity/relation storage and BFS queries."""

    def __init__(self):
        self.entities: Dict[str, KnowledgeEntity] = {}
        self.relationships: List[KnowledgeRelation] = []

    def add_entity(self, entity: KnowledgeEntity) -> str:
        """Add an entity node. Returns the entity_id."""
        self.entities[entity.entity_id] = entity
        return entity.entity_id

    def add_relationship(self, relation: KnowledgeRelation) -> None:
        """Add a directed relationship edge."""
        self.relationships.append(relation)

    def query(self, entity_id: str) -> dict:
        """Query all information about an entity and its relationships."""
        entity = self.entities.get(entity_id)
        if not entity:
            return {"error": f"Entity '{entity_id}' not found"}
        rels = [
            {
                "type": r.relation_type,
                "target": r.target_id,
                "target_name": self.entities.get(r.target_id, KnowledgeEntity("?", "?", "?")).name,
                "confidence": r.confidence,
            }
            for r in self.relationships
            if r.source_id == entity_id
        ]
        incoming = [
            {
                "type": r.relation_type,
                "source": r.source_id,
                "source_name": self.entities.get(r.source_id, KnowledgeEntity("?", "?", "?")).name,
                "confidence": r.confidence,
            }
            for r in self.relationships
            if r.target_id == entity_id
        ]
        return {
            "entity": {
                "id": entity.entity_id,
                "type": entity.entity_type,
                "name": entity.name,
                "attributes": entity.attributes,
            },
            "outgoing_relations": rels,
            "incoming_relations": incoming,
        }

    def find_path(self, start_id: str, end_id: str, max_depth: int = 5) -> list:
        """BFS path finding between two entities."""
        if start_id not in self.entities or end_id not in self.entities:
            return []
        # Build adjacency list
        adj: Dict[str, list] = {eid: [] for eid in self.entities}
        for r in self.relationships:
            if r.source_id in adj:
                adj[r.source_id].append((r.target_id, r.relation_type))
            if r.target_id in adj:
                adj[r.target_id].append((r.source_id, r.relation_type))

        visited = {start_id}
        queue = deque([(start_id, [(start_id, None)])])
        while queue:
            current, path = queue.popleft()
            if current == end_id:
                return [
                    {
                        "entity": self.entities[eid].name,
                        "relation": rel,
                    }
                    for eid, rel in path
                ]
            if len(path) > max_depth:
                continue
            for neighbor, rel_type in adj.get(current, []):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, path + [(neighbor, rel_type)]))
        return []

    def get_statistics(self) -> dict:
        """Return graph statistics."""
        type_counts = {}
        for e in self.entities.values():
            type_counts[e.entity_type] = type_counts.get(e.entity_type, 0) + 1
        rel_counts = {}
        for r in self.relationships:
            rel_counts[r.relation_type] = rel_counts.get(r.relation_type, 0) + 1
        return {
            "total_entities": len(self.entities),
            "total_relationships": len(self.relationships),
            "entity_types": type_counts,
            "relationship_types": rel_counts,
        }


# ------------------------------------------------------------------ #
# State                                                               #
# ------------------------------------------------------------------ #

class KGState(TypedDict):
    """State for knowledge graph construction workflow."""
    messages: Annotated[list[BaseMessage], add_messages]
    input_text: str
    extracted_entities: list[dict]
    extracted_relationships: list[dict]
    knowledge_graph: Any  # ScientificKnowledgeGraph instance
    validation_results: dict


# ------------------------------------------------------------------ #
# Simulated Extraction Tools                                          #
# ------------------------------------------------------------------ #

def extract_entities_tool(text: str) -> list[dict]:
    """Simulated entity extraction from scientific text."""
    return [
        {"id": "mat_1", "type": "material", "name": "BaZrS3",
         "attributes": {"crystal_system": "orthorhombic", "space_group": "Pnma"}},
        {"id": "mat_2", "type": "material", "name": "BaZrO3",
         "attributes": {"crystal_system": "cubic", "space_group": "Pm-3m"}},
        {"id": "prop_1", "type": "property", "name": "bandgap",
         "attributes": {"value": 1.8, "unit": "eV", "measurement": "UV-Vis"}},
        {"id": "prop_2", "type": "property", "name": "absorption coefficient",
         "attributes": {"value": 1e5, "unit": "cm^-1"}},
        {"id": "method_1", "type": "method", "name": "sulfurization",
         "attributes": {"temperature_c": 900, "duration_h": 4}},
        {"id": "method_2", "type": "method", "name": "XRD characterization",
         "attributes": {"instrument": "Bruker D8"}},
        {"id": "paper_1", "type": "paper", "name": "Perera et al. 2016",
         "attributes": {"doi": "10.1021/acs.nanolett.6b01999"}},
    ]


def extract_relationships_tool(entities: list[dict]) -> list[dict]:
    """Simulated relationship extraction between entities."""
    return [
        {"source": "mat_1", "target": "prop_1", "type": "has_property",
         "confidence": 0.95, "evidence": "Measured bandgap of 1.8 eV"},
        {"source": "mat_1", "target": "prop_2", "type": "has_property",
         "confidence": 0.88, "evidence": "Strong optical absorption"},
        {"source": "mat_1", "target": "method_1", "type": "synthesized_by",
         "confidence": 0.92, "evidence": "Sulfurization of oxide precursor"},
        {"source": "mat_2", "target": "mat_1", "type": "precursor_for",
         "confidence": 0.90, "evidence": "BaZrO3 converted to BaZrS3"},
        {"source": "mat_1", "target": "method_2", "type": "characterized_by",
         "confidence": 0.97, "evidence": "Phase purity confirmed by XRD"},
        {"source": "paper_1", "target": "mat_1", "type": "reports_on",
         "confidence": 1.0, "evidence": "First report of BaZrS3 thin films"},
    ]


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def extract_node(state: KGState) -> dict:
    """Extract entities and relationships from input text."""
    entities = extract_entities_tool(state["input_text"])
    relationships = extract_relationships_tool(entities)
    return {
        "extracted_entities": entities,
        "extracted_relationships": relationships,
        "messages": [AIMessage(content=(
            f"Extracted {len(entities)} entities and {len(relationships)} relationships."
        ))],
    }


def validate_node(state: KGState) -> dict:
    """Validate extracted entities and relationships."""
    valid_entities = []
    invalid = []
    for e in state["extracted_entities"]:
        if e.get("id") and e.get("type") and e.get("name"):
            valid_entities.append(e)
        else:
            invalid.append(e)

    valid_rels = []
    entity_ids = {e["id"] for e in valid_entities}
    for r in state["extracted_relationships"]:
        if r["source"] in entity_ids and r["target"] in entity_ids:
            valid_rels.append(r)

    return {
        "extracted_entities": valid_entities,
        "extracted_relationships": valid_rels,
        "validation_results": {
            "entities_valid": len(valid_entities),
            "entities_invalid": len(invalid),
            "relationships_valid": len(valid_rels),
        },
        "messages": [AIMessage(content=(
            f"Validation: {len(valid_entities)} valid entities, "
            f"{len(valid_rels)} valid relationships."
        ))],
    }


def build_graph_node(state: KGState) -> dict:
    """Build the knowledge graph from validated entities and relationships."""
    kg = ScientificKnowledgeGraph()

    for e in state["extracted_entities"]:
        kg.add_entity(KnowledgeEntity(
            entity_id=e["id"],
            entity_type=e["type"],
            name=e["name"],
            attributes=e.get("attributes", {}),
        ))

    for r in state["extracted_relationships"]:
        kg.add_relationship(KnowledgeRelation(
            source_id=r["source"],
            target_id=r["target"],
            relation_type=r["type"],
            confidence=r.get("confidence", 1.0),
            evidence=r.get("evidence", []),
        ))

    return {
        "knowledge_graph": kg,
        "messages": [AIMessage(content=(
            f"Knowledge graph built. Stats: {json.dumps(kg.get_statistics())}"
        ))],
    }


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_knowledge_graph_workflow():
    """Build extraction -> validation -> graph construction workflow."""
    workflow = StateGraph(KGState)
    workflow.add_node("extract", extract_node)
    workflow.add_node("validate", validate_node)
    workflow.add_node("build_graph", build_graph_node)

    workflow.set_entry_point("extract")
    workflow.add_edge("extract", "validate")
    workflow.add_edge("validate", "build_graph")
    workflow.add_edge("build_graph", END)

    return workflow.compile()


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

graph = build_knowledge_graph_workflow()

sample_abstract = """
BaZrS3 is a chalcogenide perovskite with a bandgap of approximately 1.8 eV,
making it a promising candidate for photovoltaic applications. The material
is synthesized by sulfurization of BaZrO3 precursor films at 900 C for 4 hours
under CS2 atmosphere. XRD characterization confirms the orthorhombic Pnma
phase with high phase purity. The material exhibits a strong absorption
coefficient of ~10^5 cm^-1. First reported by Perera et al. (2016), BaZrS3
thin films represent an earth-abundant alternative to lead halide perovskites.
"""

initial_state = {
    "messages": [HumanMessage(content="Build knowledge graph from scientific abstract")],
    "input_text": sample_abstract,
    "extracted_entities": [],
    "extracted_relationships": [],
    "knowledge_graph": None,
    "validation_results": {},
}

result = graph.invoke(initial_state)

kg = result["knowledge_graph"]
print("=" * 70)
print("KNOWLEDGE GRAPH CONSTRUCTION RESULTS")
print("=" * 70)
stats = kg.get_statistics()
print(f"\nGraph statistics: {json.dumps(stats, indent=2)}")

print("\nQuery: BaZrS3 (mat_1)")
q = kg.query("mat_1")
print(f"  Entity: {q['entity']['name']} ({q['entity']['type']})")
print(f"  Outgoing relations: {len(q['outgoing_relations'])}")
for r in q["outgoing_relations"]:
    print(f"    -> {r['type']} -> {r['target_name']} (conf: {r['confidence']})")

print("\nPath: BaZrO3 -> bandgap")
path = kg.find_path("mat_2", "prop_1")
if path:
    print("  " + " -> ".join(f"{p['entity']}" + (f" [{p['relation']}]" if p['relation'] else "") for p in path))
else:
    print("  No path found")


---

## Listing 7-5: Cross-Disciplinary Research Coordinator

Coordinates research across multiple scientific domains by gathering domain-specific knowledge, discovering cross-domain connections, and synthesizing collaboration opportunities.

In [ ]:
"""
Listing 7-5: Cross-Disciplinary Research Coordinator

Gathers knowledge from multiple research domains, identifies cross-domain
connections, and synthesizes collaboration opportunities. All domain
searches are simulated.
"""

from typing import TypedDict, Annotated, List, Dict, Any
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json


# ------------------------------------------------------------------ #
# State                                                               #
# ------------------------------------------------------------------ #

class CoordinatorState(TypedDict):
    """State for cross-disciplinary research coordination."""
    messages: Annotated[list[BaseMessage], add_messages]
    domains: list[str]
    domain_summaries: dict
    connections: list[dict]
    synthesis: dict


# ------------------------------------------------------------------ #
# Domain Knowledge Profiles                                           #
# ------------------------------------------------------------------ #

DOMAIN_PROFILES = {
    "materials_science": {
        "domain": "Materials Science",
        "key_methods": [
            "DFT calculations", "molecular dynamics", "high-throughput screening",
            "combinatorial synthesis", "in-situ characterization",
        ],
        "current_challenges": [
            "Predicting properties of multi-component alloys",
            "Accelerating materials discovery cycle",
            "Bridging simulation-experiment gap",
        ],
        "recent_breakthroughs": [
            "GNoME: 2.2M new stable crystal structures predicted (2023)",
            "Autonomous labs for battery materials optimization",
        ],
        "data_resources": ["Materials Project", "AFLOW", "OQMD", "NOMAD"],
    },
    "machine_learning": {
        "domain": "Machine Learning",
        "key_methods": [
            "Graph neural networks", "transformers", "active learning",
            "Bayesian optimization", "few-shot learning",
        ],
        "current_challenges": [
            "Data efficiency for scientific applications",
            "Uncertainty quantification in predictions",
            "Interpretability of deep models",
        ],
        "recent_breakthroughs": [
            "Foundation models for molecular property prediction",
            "Diffusion models for crystal structure generation",
        ],
        "data_resources": ["Open Catalyst", "QM9", "PubChem", "UniProt"],
    },
    "chemistry": {
        "domain": "Chemistry",
        "key_methods": [
            "Retrosynthesis planning", "reaction optimization",
            "spectroscopic analysis", "computational chemistry",
        ],
        "current_challenges": [
            "Sustainable synthesis routes",
            "Scaling laboratory results to production",
            "Predicting reaction selectivity",
        ],
        "recent_breakthroughs": [
            "AI-driven retrosynthesis (ASKCOS)",
            "Autonomous chemical synthesis platforms (ChemCrow)",
        ],
        "data_resources": ["Reaxys", "SciFinder", "CSD", "PDB"],
    },
}


# ------------------------------------------------------------------ #
# Simulated Tools                                                     #
# ------------------------------------------------------------------ #

def search_domain(domain: str) -> dict:
    """Simulated domain knowledge retrieval."""
    return DOMAIN_PROFILES.get(domain, {
        "domain": domain,
        "key_methods": ["General research methods"],
        "current_challenges": ["Domain-specific challenges"],
        "recent_breakthroughs": [],
        "data_resources": [],
    })


def find_connections(domain_a: str, domain_b: str) -> list[dict]:
    """Simulated cross-domain connection discovery."""
    connections_db = {
        ("materials_science", "machine_learning"): [
            {
                "connection_type": "method_transfer",
                "description": "Graph neural networks for crystal property prediction",
                "strength": 0.92,
                "examples": ["MEGNet", "CGCNN", "SchNet", "DimeNet"],
                "opportunity": "Apply latest GNN architectures to multi-component alloy prediction",
            },
            {
                "connection_type": "shared_challenge",
                "description": "Uncertainty quantification in property predictions",
                "strength": 0.85,
                "examples": ["Ensemble methods", "MC Dropout", "Evidential deep learning"],
                "opportunity": "Develop calibrated UQ methods for materials screening",
            },
        ],
        ("materials_science", "chemistry"): [
            {
                "connection_type": "workflow_integration",
                "description": "Computational screening to synthesis planning pipeline",
                "strength": 0.88,
                "examples": ["DFT -> retrosynthesis", "Active learning + synthesis"],
                "opportunity": "End-to-end discovery: predict, plan synthesis, execute",
            },
        ],
        ("machine_learning", "chemistry"): [
            {
                "connection_type": "method_transfer",
                "description": "Transformer models for reaction prediction",
                "strength": 0.90,
                "examples": ["Molecular Transformer", "ReactionT5"],
                "opportunity": "Fine-tune foundation models on domain-specific reactions",
            },
        ],
    }
    key = tuple(sorted([domain_a, domain_b]))
    return connections_db.get(key, [{
        "connection_type": "potential",
        "description": f"Unexplored connection between {domain_a} and {domain_b}",
        "strength": 0.3,
        "examples": [],
        "opportunity": "Investigate shared methodology and data formats",
    }])


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def gather_domain_knowledge_node(state: CoordinatorState) -> dict:
    """Gather knowledge summaries for each research domain."""
    summaries = {}
    for domain in state["domains"]:
        summaries[domain] = search_domain(domain)
    return {
        "domain_summaries": summaries,
        "messages": [AIMessage(content=(
            f"Gathered knowledge for {len(summaries)} domains: "
            f"{', '.join(d for d in summaries)}"
        ))],
    }


def discover_connections_node(state: CoordinatorState) -> dict:
    """Discover connections between all pairs of domains."""
    domains = state["domains"]
    all_connections = []
    for i in range(len(domains)):
        for j in range(i + 1, len(domains)):
            conns = find_connections(domains[i], domains[j])
            for c in conns:
                c["domains"] = [domains[i], domains[j]]
            all_connections.extend(conns)
    all_connections.sort(key=lambda c: c.get("strength", 0), reverse=True)
    return {
        "connections": all_connections,
        "messages": [AIMessage(content=(
            f"Discovered {len(all_connections)} cross-domain connections."
        ))],
    }


def synthesize_node(state: CoordinatorState) -> dict:
    """Synthesize findings into collaboration recommendations."""
    strong = [c for c in state["connections"] if c.get("strength", 0) > 0.8]
    synthesis = {
        "total_connections": len(state["connections"]),
        "strong_connections": len(strong),
        "top_opportunities": [
            {
                "domains": c["domains"],
                "opportunity": c["opportunity"],
                "strength": c["strength"],
                "type": c["connection_type"],
            }
            for c in strong[:5]
        ],
        "recommended_actions": [
            "Form joint working group across top-connected domains",
            "Establish shared data repository with standardized formats",
            "Schedule monthly cross-domain seminars",
            "Identify pilot project combining strongest connection areas",
        ],
        "shared_data_resources": list({
            r
            for s in state["domain_summaries"].values()
            for r in s.get("data_resources", [])
        }),
    }
    return {
        "synthesis": synthesis,
        "messages": [AIMessage(content="Synthesis complete. Collaboration plan generated.")],
    }


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_coordinator_graph():
    """Build domain knowledge -> connection discovery -> synthesis workflow."""
    workflow = StateGraph(CoordinatorState)
    workflow.add_node("gather_knowledge", gather_domain_knowledge_node)
    workflow.add_node("discover_connections", discover_connections_node)
    workflow.add_node("synthesize", synthesize_node)

    workflow.set_entry_point("gather_knowledge")
    workflow.add_edge("gather_knowledge", "discover_connections")
    workflow.add_edge("discover_connections", "synthesize")
    workflow.add_edge("synthesize", END)

    return workflow.compile()


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

graph = build_coordinator_graph()

initial_state = {
    "messages": [HumanMessage(content="Coordinate research across materials science, ML, and chemistry")],
    "domains": ["materials_science", "machine_learning", "chemistry"],
    "domain_summaries": {},
    "connections": [],
    "synthesis": {},
}

result = graph.invoke(initial_state)

print("=" * 70)
print("CROSS-DISCIPLINARY COORDINATION RESULTS")
print("=" * 70)
syn = result["synthesis"]
print(f"\nDomains analyzed: {', '.join(result['domains'])}")
print(f"Total connections found: {syn['total_connections']}")
print(f"Strong connections (>0.8): {syn['strong_connections']}")
print(f"\nTop opportunities:")
for opp in syn["top_opportunities"]:
    print(f"  [{opp['strength']}] {' + '.join(opp['domains'])}: {opp['opportunity']}")
print(f"\nShared data resources: {', '.join(sorted(syn['shared_data_resources']))}")
print(f"\nRecommended actions:")
for a in syn["recommended_actions"]:
    print(f"  - {a}")
